## LAB 7 - Clock Domain Crossing (CDC) Pattern

In [6]:
import os, random

HERE = os.path.join(os.getcwd(), "00_TESTBED")
os.makedirs(HERE, exist_ok=True)
print("HERE =", HERE)

ID_MIN, ID_MAX = 0, 31
SCORE_MIN, SCORE_MAX = 50, 200
W_MIN, W_MAX = 0, 7
NUM_DOOR = 5

HERE = c:\Users\harry\Desktop\Master\Training-Project\CDC\00_TESTBED


### Generate Pattern

In [ ]:
def pref_score(d, sw, iw, ew):
    return d["size"] * sw + d["iq"] * iw + d["eq"] * ew

def gen_pattern(patnum=1000, seed=42, here=None):
    here = here or HERE
    rng = random.Random(seed)

    def rand_d():
        return {"id": rng.randint(ID_MIN, ID_MAX),
                "size": rng.randint(SCORE_MIN, SCORE_MAX),
                "iq": rng.randint(SCORE_MIN, SCORE_MAX),
                "eq": rng.randint(SCORE_MIN, SCORE_MAX)}

    in_lines = [str(patnum)]
    out_lines = []
    doors = [None] * NUM_DOOR

    for door in range(4):
        d = rand_d(); doors[door] = d
        in_lines.append(f"{d['id']} {d['size']} {d['iq']} {d['eq']}")

    prev = None
    for k in range(patnum):
        d = rand_d()
        sw = rng.randint(W_MIN, W_MAX)
        iw = rng.randint(W_MIN, W_MAX)
        ew = rng.randint(W_MIN, W_MAX)
        target = 4 if k == 0 else prev      
        doors[target] = d
        in_lines.append(f"{d['id']} {d['size']} {d['iq']} {d['eq']} {sw} {iw} {ew}")

        best, bs = 0, pref_score(doors[0], sw, iw, ew)
        for door in range(1, NUM_DOOR):         
            s = pref_score(doors[door], sw, iw, ew)
            if s > bs:
                bs, best = s, door
        out_lines.append(f"{best} {doors[best]['id']}")
        prev = best

    with open(os.path.join(here, "input.txt"), "w", newline="\n") as f:
        f.write("\n".join(in_lines) + "\n")
    with open(os.path.join(here, "output.txt"), "w", newline="\n") as f:
        f.write("\n".join(out_lines) + "\n")

    print(f"PATNUM={patnum} seed={seed}  ->  input.txt={len(in_lines)} lines, output.txt={len(out_lines)} lines")

### Generate Debug

In [8]:
def _load_inputs(here):
    toks = open(os.path.join(here, "input.txt")).read().split()
    p = [0]
    def g(n):
        v = [int(x) for x in toks[p[0]:p[0] + n]]; p[0] += n; return v
    patnum = g(1)[0]
    doors = [g(4) for _ in range(4)] + [[0, 0, 0, 0]]
    rounds = [(g(4), g(3)) for _ in range(patnum)]
    return patnum, doors, rounds

def _load_golden(here):
    t = open(os.path.join(here, "output.txt")).read().split()
    return [(int(t[2 * i]), int(t[2 * i + 1])) for i in range(len(t) // 2)]

def _round_block(k, doors, dora, tgt, scores, best, golden):
    bid = doors[best][0]; out8 = (best << 5) | bid
    g = golden[k] if k < len(golden) else None
    ok = (g == (best, bid))
    L = [f"=== No.{k}  (output.txt line {k+1}, input.txt line {6+k}) ===",
         f"    new doraemon: id={dora[0]} size={dora[1]} iq={dora[2]} eq={dora[3]}  -> fill door{tgt}",
         f"    weights: size_w={scores[5]} iq_w={scores[6]} eq_w={scores[7]}"]
    for d in range(NUM_DOOR):
        tag = " *NEW*" if d == tgt else ""
        win = " <-- WINNER" if d == best else ""
        L.append(f"      door{d}: id={doors[d][0]:2d} size={doors[d][1]:3d} iq={doors[d][2]:3d} eq={doors[d][3]:3d} | score={scores[d]:6d}{tag}{win}")
    gstr = f'"{g[0]} {g[1]}"' if g else "N/A"
    L.append(f'    => out = door{best} id{bid} = "{best} {bid}"  (8b={out8:08b} = {out8})   golden={gstr}   {"OK" if ok else "MISMATCH!"}')
    return L, ok

def gen_debug(out_name="debug.txt", here=None):
    here = here or HERE
    patnum, doors, rounds = _load_inputs(here)
    golden = _load_golden(here)
    mism = 0; sel = None; lines = []
    for k in range(patnum):
        dora, w = rounds[k]; sw, iw, ew = w
        tgt = 4 if k == 0 else sel
        doors[tgt] = dora
        scores = [doors[d][1]*sw + doors[d][2]*iw + doors[d][3]*ew for d in range(NUM_DOOR)] + [sw, iw, ew]
        best = 0
        for d in range(1, NUM_DOOR):
            if scores[d] > scores[best]:
                best = d
        block, ok = _round_block(k, doors, dora, tgt, scores, best, golden)
        if not ok: mism += 1
        lines += block + [""]
        sel = best
    lines += ["=" * 60,
              f"PATNUM={patnum}  rounds_written={patnum}  golden_lines={len(golden)}  mismatches={mism}"]
    with open(os.path.join(here, out_name), "w", newline="\n") as f:
        f.write("\n".join(lines) + "\n")
    print(f"wrote {os.path.join(here, out_name)}")
    print(f"PATNUM={patnum}, mismatches={mism}")

In [9]:
gen_pattern(patnum=6000, seed=42)
gen_debug()

PATNUM=6000 seed=42  ->  input.txt=6005 lines, output.txt=6000 lines
wrote c:\Users\harry\Desktop\Master\Training-Project\CDC\00_TESTBED\debug.txt
PATNUM=6000, mismatches=0
